# Q-Shield: Siamese Network Training v3 — Surgical Fix

**Post-mortem v2:** Over-regularized + undertrained + wrong augmentation.
**v3 Strategy:** Middle-ground regularization, longer training with warm restarts, focal loss, stronger head.

### Key changes from v2
| Param | v2 | v3 | Why |
|-------|-----|-----|-----|
| Dropout | 0.5 | 0.35 | v2 under-fit |
| Weight decay | 5e-4 | 2e-4 | Less restrictive |
| Margin | 1.0 | 1.5 | More separation |
| Epochs P1 | 25 | 40 | v2 didn't converge |
| LR schedule | cosine → 0 | cosine **warm restart** | Escape plateaus |
| Rotation aug | ±180° | **Only H-flip** | QR has semantic orientation |
| Classifier head | 256→64→1 | **512→128→32→1** | More capacity |
| Phase 2 loss | BCE | **Focal Loss (γ=2)** | Penalize false negatives |
| Phase 2 start | unfrozen | **5 ep frozen → unfreeze** | Stabilize head first |

**Target:** AUC ≥ 0.91, FNR ≤ 15%
**Author:** Nicolas A. Llerena Silva (UTEC)

In [ ]:
# 0. SETUP

import sys, os, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Proyecto_Quishing_Detection_Nicolas'
assert os.path.exists(BASE), f'BASE not found: {BASE}'
print(f'Drive: {BASE}')

!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from PIL import Image
import pickle, zipfile, random, time, copy, json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram/1e9:.1f} GB')

In [ ]:
# 1. LOAD DATA (same as v2 — already works)

WORK = '/content/qshield_data'
os.makedirs(WORK, exist_ok=True)

# Trad
trad_dir = os.path.join(WORK, 'trad')
if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip')) as z:
        z.extractall(trad_dir)
with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

# CIC
cic_b_dir = os.path.join(WORK, 'cic_benign')
cic_m_dir = os.path.join(WORK, 'cic_malicious')
zip_b = os.path.join(BASE, 'QR_benign_430K.zip')
zip_m = os.path.join(BASE, 'QR_malicious_576K.zip')
HAS_CIC = os.path.exists(zip_b) and os.path.exists(zip_m)

if HAS_CIC:
    if not os.path.exists(cic_b_dir) or len(os.listdir(cic_b_dir)) == 0:
        os.makedirs(cic_b_dir, exist_ok=True)
        print('Extracting CIC benign...')
        with zipfile.ZipFile(zip_b) as z: z.extractall(cic_b_dir)
    if not os.path.exists(cic_m_dir) or len(os.listdir(cic_m_dir)) == 0:
        os.makedirs(cic_m_dir, exist_ok=True)
        print('Extracting CIC malicious...')
        with zipfile.ZipFile(zip_m) as z: z.extractall(cic_m_dir)

    cic_b_files = sorted(glob.glob(os.path.join(cic_b_dir, '**', '*.png'), recursive=True))
    cic_m_files = sorted(glob.glob(os.path.join(cic_m_dir, '**', '*.png'), recursive=True))
    random.seed(SEED)
    cic_b_files = random.sample(cic_b_files, min(50000, len(cic_b_files)))
    cic_m_files = random.sample(cic_m_files, min(50000, len(cic_m_files)))
    print(f'Data ready: Trad={len(trad_qr):,}, CIC benign={len(cic_b_files):,}, CIC mal={len(cic_m_files):,}')
else:
    cic_b_files, cic_m_files = [], []
    print('CIC not found')

---
## 2. MODEL (dropout 0.35, better head)

In [ ]:
# 2.1 MODEL v3

class MobileNetV2Embedding(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        mn = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        orig = mn.features[0][0]
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(orig.weight.mean(dim=1, keepdim=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(512, emb_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.projection(x)
        return F.normalize(x, p=2, dim=1)

class SiameseQRNet(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        self.backbone = MobileNetV2Embedding(emb_dim, pretrained, dropout)

    def forward_one(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        return self.backbone(x1), self.backbone(x2)

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.5):
        super().__init__()
        self.margin = margin

    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        return ((1-y)*0.5*d.pow(2) + y*0.5*F.relu(self.margin-d).pow(2)).mean()

class FocalLoss(nn.Module):
    """Focal Loss for binary classification (Lin et al. 2017).
    Penalizes hard examples more, reducing false negatives."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce
        return loss.mean()

model = SiameseQRNet(emb_dim=128, pretrained=True, dropout=0.35).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

---
## 3. DATASETS (only H-flip, no rotation)

In [ ]:
# 3.1 DATASETS

# v3: ONLY horizontal flip. No rotation (QR codes have semantic orientation)
train_aug = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
])

class TradPairDataset(Dataset):
    def __init__(self, qr, labels, n_pairs, augment=False):
        self.qr = qr.astype(np.float32)
        self.labels = np.array(labels)
        self.n = n_pairs
        self.idx = {0: np.where(self.labels==0)[0], 1: np.where(self.labels==1)[0]}
        self.aug = train_aug if augment else None

    def __len__(self): return self.n

    def _tensor(self, i):
        t = torch.from_numpy(self.qr[i]).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        return self.aug(t) if self.aug else t

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.choice(self.idx[c])
        c2 = c if same else 1-c; i2 = random.choice(self.idx[c2])
        return self._tensor(i1), self._tensor(i2), torch.tensor(0.0 if same else 1.0)

class CICPairDataset(Dataset):
    def __init__(self, benign, mal, n_pairs, augment=False):
        self.files = {0: benign, 1: mal}
        self.n = n_pairs
        self.aug = train_aug if augment else None

    def __len__(self): return self.n

    def _load(self, cls, idx):
        img = Image.open(self.files[cls][idx]).convert('L').resize((224,224))
        arr = np.array(img, dtype=np.float32)/255.0
        t = torch.from_numpy(arr).unsqueeze(0)
        return self.aug(t) if self.aug else t

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.randint(0, len(self.files[c])-1)
        c2 = c if same else 1-c; i2 = random.randint(0, len(self.files[c2])-1)
        return self._load(c,i1), self._load(c2,i2), torch.tensor(0.0 if same else 1.0)

class ClassifyDataset(Dataset):
    def __init__(self, trad_qr=None, trad_labels=None, cic_b=None, cic_m=None, augment=False):
        self.items = []
        if trad_qr is not None:
            for i in range(len(trad_qr)):
                self.items.append(('trad', i, int(trad_labels[i])))
            self.trad_qr = trad_qr.astype(np.float32)
        if cic_b is not None:
            for i,_ in enumerate(cic_b): self.items.append(('cic_b', i, 0))
            for i,_ in enumerate(cic_m): self.items.append(('cic_m', i, 1))
            self.cic_b = cic_b; self.cic_m = cic_m
        self.aug = train_aug if augment else None

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        src, i, lbl = self.items[idx]
        if src == 'trad':
            t = torch.from_numpy(self.trad_qr[i]).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        else:
            files = self.cic_b if src == 'cic_b' else self.cic_m
            img = Image.open(files[i]).convert('L').resize((224,224))
            t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0)
        if self.aug: t = self.aug(t)
        return t, torch.tensor(float(lbl))

# Splits
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=SEED)
qr_tr, lab_tr = trad_qr[idx_tr], trad_labels[idx_tr]
qr_val, lab_val = trad_qr[idx_val], trad_labels[idx_val]

if HAS_CIC:
    sb = int(len(cic_b_files)*0.8); sm = int(len(cic_m_files)*0.8)
    cic_b_tr, cic_b_val = cic_b_files[:sb], cic_b_files[sb:]
    cic_m_tr, cic_m_val = cic_m_files[:sm], cic_m_files[sm:]

BATCH = 128
PAIRS_TR = 60000; PAIRS_VAL = 8000
NUM_WORKERS = 4 if IN_COLAB else 0

if HAS_CIC:
    train_ds = ConcatDataset([
        TradPairDataset(qr_tr, lab_tr, PAIRS_TR, augment=True),
        CICPairDataset(cic_b_tr, cic_m_tr, PAIRS_TR, augment=True),
    ])
    val_ds = ConcatDataset([
        TradPairDataset(qr_val, lab_val, PAIRS_VAL, augment=False),
        CICPairDataset(cic_b_val, cic_m_val, PAIRS_VAL, augment=False),
    ])
else:
    train_ds = TradPairDataset(qr_tr, lab_tr, PAIRS_TR, augment=True)
    val_ds = TradPairDataset(qr_val, lab_val, PAIRS_VAL, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print(f'Train pairs: {len(train_ds):,}  Val pairs: {len(val_ds):,}')

---
## 4. PHASE 1 — Contrastive with warm restarts

In [ ]:
# 4. PHASE 1 TRAINING (40 epochs, warm restart every 15)

EMB_DIM = 128
MARGIN = 1.5
LR1 = 2e-4
EPOCHS1 = 40
WD = 2e-4

model = SiameseQRNet(emb_dim=EMB_DIM, pretrained=True, dropout=0.35).to(device)
criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.AdamW(model.parameters(), lr=LR1, weight_decay=WD)
# Warm restarts: every 15 epochs LR restarts to original
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=1, eta_min=1e-6)

hist1 = {'tl':[], 'vl':[], 'ta':[], 'va':[]}
best_vl = float('inf'); best_state = None; patience = 0
PATIENCE_LIMIT = 8  # allow longer wait because of restarts

print(f'v3 Config: margin={MARGIN}, LR={LR1}, WD={WD}, dropout=0.35, restarts every 15 ep')
print(f'{"Ep":>3} {"TrLoss":>8} {"VaLoss":>8} {"TrAcc":>7} {"VaAcc":>7} {"LR":>10} {"Time":>6}')
print('-'*58)

for ep in range(1, EPOCHS1+1):
    t0 = time.time()
    model.train()
    tl, tc, tt = 0, 0, 0
    for x1, x2, y in train_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad()
        e1, e2 = model(x1, x2)
        loss = criterion(e1, e2, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tl += loss.item()*x1.size(0)
        with torch.no_grad():
            d = F.pairwise_distance(e1, e2)
            tc += ((d > MARGIN/2).float() == y).sum().item()
            tt += y.size(0)

    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for x1, x2, y in val_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            e1, e2 = model(x1, x2)
            vl += criterion(e1, e2, y).item()*x1.size(0)
            d = F.pairwise_distance(e1, e2)
            vc += ((d > MARGIN/2).float() == y).sum().item()
            vt += y.size(0)

    scheduler.step()
    tl_a, vl_a = tl/tt, vl/vt; ta, va = tc/tt, vc/vt
    hist1['tl'].append(tl_a); hist1['vl'].append(vl_a); hist1['ta'].append(ta); hist1['va'].append(va)

    mk = ''
    if vl_a < best_vl:
        best_vl = vl_a; best_state = copy.deepcopy(model.state_dict()); patience = 0; mk = ' *'
    else:
        patience += 1

    lr = optimizer.param_groups[0]['lr']
    dt = time.time()-t0
    print(f'{ep:>3} {tl_a:>8.4f} {vl_a:>8.4f} {ta:>6.1%} {va:>6.1%} {lr:>10.6f} {dt:>5.0f}s{mk}')

    if patience >= PATIENCE_LIMIT:
        print(f'\nEarly stop at epoch {ep}')
        break

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(BASE, 'siamese_v3_phase1.pth'))
print(f'\nBest val loss: {best_vl:.4f}')

---
## 5. PHASE 2 — Bigger head, focal loss, progressive unfreeze

In [ ]:
# 5. PHASE 2 TRAINING — focal loss + frozen start

class QRClassifier(nn.Module):
    """v3: deeper head with more capacity"""
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def set_backbone_grad(self, requires_grad):
        for p in self.backbone.parameters():
            p.requires_grad = requires_grad

    def forward(self, x):
        return self.head(self.backbone(x))

classifier = QRClassifier(model.backbone, EMB_DIM).to(device)

tr_cls = ClassifyDataset(trad_qr=qr_tr, trad_labels=lab_tr,
                         cic_b=cic_b_tr if HAS_CIC else None,
                         cic_m=cic_m_tr if HAS_CIC else None, augment=True)
val_cls = ClassifyDataset(trad_qr=qr_val, trad_labels=lab_val,
                          cic_b=cic_b_val if HAS_CIC else None,
                          cic_m=cic_m_val if HAS_CIC else None, augment=False)
tr_loader = DataLoader(tr_cls, batch_size=256, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True)
va_loader = DataLoader(val_cls, batch_size=512, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True)
print(f'Phase 2: {len(tr_cls):,} train, {len(val_cls):,} val')

EPOCHS2 = 20
FROZEN_EP = 5  # first 5 epochs: freeze backbone
focal = FocalLoss(alpha=0.5, gamma=2.0)  # alpha=0.5 (balanced) with focal effect

# Phase 2a: frozen backbone (only train head)
classifier.set_backbone_grad(False)
opt2 = optim.AdamW([p for p in classifier.parameters() if p.requires_grad],
                   lr=5e-4, weight_decay=1e-4)  # higher LR for head-only

hist2 = {'tl':[], 'vl':[], 'auc':[], 'f1':[]}
best_auc = 0; best_cls = None

print(f'{"Ep":>3} {"Phase":<12} {"TrLoss":>8} {"VaLoss":>8} {"AUC":>7} {"F1":>7} {"Prec":>7} {"Rec":>7}')
print('-'*74)

for ep in range(1, EPOCHS2+1):
    # Unfreeze at epoch 6
    if ep == FROZEN_EP + 1:
        classifier.set_backbone_grad(True)
        opt2 = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=2e-4)
        sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS2-FROZEN_EP, eta_min=1e-6)
        phase = 'unfrozen'
    elif ep > FROZEN_EP:
        phase = 'unfrozen'
    else:
        phase = 'frozen'

    classifier.train()
    tl = 0; n = 0
    for imgs, lbls in tr_loader:
        imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
        opt2.zero_grad()
        loss = focal(classifier(imgs), lbls)
        loss.backward()
        opt2.step()
        tl += loss.item()*imgs.size(0); n += imgs.size(0)

    classifier.eval()
    vl = 0; probs, true = [], []; nv = 0
    with torch.no_grad():
        for imgs, lbls in va_loader:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            logits = classifier(imgs)
            vl += focal(logits, lbls).item()*imgs.size(0); nv += imgs.size(0)
            probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            true.extend(lbls.cpu().numpy().flatten())

    if ep > FROZEN_EP:
        sch2.step()

    probs, true = np.array(probs), np.array(true)
    preds = (probs >= 0.5).astype(int)
    auc = roc_auc_score(true, probs)
    f1 = f1_score(true, preds); prec = precision_score(true, preds); rec = recall_score(true, preds)
    hist2['tl'].append(tl/n); hist2['vl'].append(vl/nv); hist2['auc'].append(auc); hist2['f1'].append(f1)

    mk = ''
    if auc > best_auc:
        best_auc = auc; best_cls = copy.deepcopy(classifier.state_dict()); mk = ' *'

    print(f'{ep:>3} {phase:<12} {tl/n:>8.4f} {vl/nv:>8.4f} {auc:>6.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f}{mk}')

classifier.load_state_dict(best_cls)
torch.save(best_cls, os.path.join(BASE, 'classifier_v3_phase2.pth'))
print(f'\nBest AUC: {best_auc:.4f}')

In [ ]:
# 6. FINAL EVALUATION

classifier.eval()
probs, true = [], []
with torch.no_grad():
    for imgs, lbls in va_loader:
        logits = classifier(imgs.to(device))
        probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        true.extend(lbls.numpy().flatten())
probs, true = np.array(probs), np.array(true)
preds = (probs >= 0.5).astype(int)
final_auc = roc_auc_score(true, probs)
final_f1 = f1_score(true, preds)

print('='*60)
print(f' Q-Shield v3 FINAL — AUC={final_auc:.4f}, F1={final_f1:.4f}')
print('='*60)
print(classification_report(true, preds, target_names=['Benign', 'Phishing']))

cm = confusion_matrix(true, preds)
fnr = cm[1,0] / cm[1].sum()
print(f'\nFalse Negative Rate: {fnr:.4f} (target <= 0.15)')

fpr, tpr, _ = roc_curve(true, probs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion (AUC={final_auc:.4f}, FNR={fnr:.2%})', fontweight='bold')
axes[1].plot(fpr, tpr, lw=2, color='#e74c3c', label=f'v3 (AUC={final_auc:.4f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('ROC', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle('Q-Shield v3 Final', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_v3_final.png'), dpi=300, bbox_inches='tight')
plt.show()

# Comparison
print('\n' + '='*70)
print('Q-SHIELD PROGRESSION')
print('='*70)
comp = pd.DataFrame([
    {'Method':'Trad et al. (SOTA)', 'AUC':0.9133, 'F1':0.89, 'FNR':'-'},
    {'Method':'Ours: 25 feat + RF', 'AUC':0.8132, 'F1':0.72, 'FNR':'0.34'},
    {'Method':'Ours: Siamese v1', 'AUC':0.8860, 'F1':0.81, 'FNR':'0.21'},
    {'Method':'Ours: Siamese v2', 'AUC':0.8678, 'F1':0.78, 'FNR':'0.27'},
    {'Method':'Ours: Siamese v3', 'AUC':round(final_auc,4), 'F1':round(final_f1,4), 'FNR':f'{fnr:.2f}'},
])
print(comp.to_string(index=False))
print(f'\nv3 vs Trad: {final_auc - 0.9133:+.4f}')
print(f'v3 vs v1:   {final_auc - 0.886:+.4f}')
print(f'v3 vs v2:   {final_auc - 0.8678:+.4f}')

results = {'version':'v3', 'phase1':hist1, 'phase2':hist2,
           'final_auc':final_auc, 'final_f1':final_f1, 'fnr':fnr,
           'confusion_matrix':cm.tolist()}
with open(os.path.join(BASE, 'experiment_results_v3.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to Drive.')